# LLN & CLT — companion notebook

Runnable companion for [`clt-lln.md`](./clt-lln.md).

1. Sample means of various distributions converging to the true mean (LLN).
2. Histograms of the standardized sample mean across base distributions converging to $\mathcal{N}(0, 1)$ (CLT).
3. A heavy-tailed counterexample (Cauchy) where the CLT fails.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

## 1. LLN: sample means converge to the true mean

Plot running averages $\bar X_n$ for several base distributions; they all settle to their $\mu$.

In [ ]:
N = 5000
samplers = {
    'Uniform[0,1]  (mu=0.5)':  (rng.uniform(0, 1, N), 0.5),
    'Exponential(1) (mu=1)':   (rng.exponential(1.0, N), 1.0),
    'Bernoulli(0.3) (mu=0.3)': (rng.binomial(1, 0.3, N).astype(float), 0.3),
    'Poisson(4)    (mu=4)':    (rng.poisson(4, N).astype(float), 4.0),
}

fig, ax = plt.subplots(figsize=(9, 5))
ns = np.arange(1, N + 1)
for label, (xs, mu) in samplers.items():
    running = np.cumsum(xs) / ns
    line, = ax.plot(ns, running, label=label, lw=1)
    ax.axhline(mu, color=line.get_color(), ls=':', lw=1)
ax.set_xscale('log')
ax.set_xlabel('n  (log)'); ax.set_ylabel('running sample mean')
ax.set_title('LLN: running mean stabilizes at the true mean')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 2. CLT: standardized sample means look Normal

For each base distribution, draw $R$ replicate samples of size $n$, compute the standardized mean $Z = \sqrt{n}(\bar X_n - \mu)/\sigma$, and overlay $\mathcal{N}(0, 1)$. Even for highly skewed bases the histogram tightens onto the bell curve as $n$ grows.

In [ ]:
R = 4000  # replicates

def draw(name, n, R):
    if name == 'Uniform[0,1]':
        x = rng.uniform(0, 1, (R, n)); mu, sd = 0.5, np.sqrt(1/12)
    elif name == 'Exponential(1)':
        x = rng.exponential(1.0, (R, n)); mu, sd = 1.0, 1.0
    elif name == 'Bernoulli(0.2)':
        x = rng.binomial(1, 0.2, (R, n)).astype(float); mu, sd = 0.2, np.sqrt(0.2 * 0.8)
    else:
        raise ValueError(name)
    return x, mu, sd

base_names = ['Uniform[0,1]', 'Exponential(1)', 'Bernoulli(0.2)']
ns = [2, 10, 50]
zgrid = np.linspace(-4, 4, 200)
phi = stats.norm.pdf(zgrid)

fig, axes = plt.subplots(len(base_names), len(ns), figsize=(11, 8), sharex=True, sharey=True)
for i, name in enumerate(base_names):
    for j, n in enumerate(ns):
        x, mu, sd = draw(name, n, R)
        z = np.sqrt(n) * (x.mean(axis=1) - mu) / sd
        ax = axes[i, j]
        ax.hist(z, bins=40, density=True, alpha=0.6, color='tab:blue')
        ax.plot(zgrid, phi, 'tab:red', lw=2)
        ax.set_xlim(-4, 4)
        if i == 0:
            ax.set_title(f'n = {n}')
        if j == 0:
            ax.set_ylabel(name)
fig.suptitle('CLT: histogram of standardized sample mean -> N(0, 1)')
plt.tight_layout(); plt.show()

## 3. Counter-example: Cauchy (infinite variance) breaks the CLT

Standard Cauchy has no mean and infinite variance. The sample mean of $n$ iid Cauchy draws is *itself* Cauchy — no concentration, no Gaussian limit.

In [ ]:
N = 5000
x_cauchy = stats.cauchy.rvs(size=N, random_state=rng)
running = np.cumsum(x_cauchy) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(running, lw=1)
ax.set_xscale('log')
ax.set_xlabel('n (log)'); ax.set_ylabel('running mean')
ax.set_title('Cauchy sample mean does NOT settle (no LLN, no CLT)')
ax.grid(True, alpha=0.3)
plt.show()

print('Min / max of the running mean trajectory:',
      running.min().round(2), running.max().round(2))